# Multi-Cancer Dataset Expansion
**Genomic-RawSeq-Analyzer — Semester 2**

Downloads and preprocesses two new WXS cohorts from NCBI SRA (via ENA mirrors)
to test the pipeline's generalizability across cancer types:

| Cohort | Type | GEO | Samples |
|--------|------|-----|---------|
| **BRCA** | Breast Invasive Carcinoma | GSE48215 | 25 Tumor + 25 Normal |
| **LUAD** | Lung Adenocarcinoma | GSE40419 | 17 Tumor + 13 Normal |

After download, evaluates the **Semester 1 CNN** on each new cohort **zero-shot**
(no retraining) to measure cross-cancer transfer performance.

**Steps:**
1. Get the SRA run list (hardcoded subset or uploaded SraRunTable)
2. Download FASTQ files directly from ENA over HTTPS (partial range-fetch, no SRA-toolkit)
3. Preprocess: integer-encode reads and save batches
4. Check class balance
5. Run zero-shot CNN evaluation (read-level + patient-level AUC)

**Outputs saved to Google Drive:**
- `results/multi_cancer/brca/` — batch .npz files
- `results/multi_cancer/luad/` — batch .npz files
- `results/multi_cancer/brca/zero_shot_eval/`
- `results/multi_cancer/luad/zero_shot_eval/`

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, glob
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, precision_recall_fscore_support

!pip install -q tensorflow biopython scikit-learn matplotlib requests
from tensorflow.keras.models import load_model

BASE = '/content/drive/MyDrive/DNA_Anomaly_Detection'

# ── Inline: load_all_batches ──────────────────────────────────────────
def load_all_batches(batch_dir):
    files = sorted(glob.glob(os.path.join(batch_dir, 'batch_*.npz')))
    if not files:
        raise FileNotFoundError(f'No batch_*.npz files in {batch_dir}')
    X_parts, y_parts, id_parts = [], [], []
    for f in files:
        print(f'Loading {os.path.basename(f)}...')
        with np.load(f, allow_pickle=True) as d:
            X_parts.append(d['X'])
            y_parts.append(d['y'])
            if 'run_ids' in d:
                id_parts.append(d['run_ids'])
            else:
                n = len(d['X'])
                name = os.path.basename(f).replace('.npz', '')
                id_parts.append(np.array([f'{name}_read_{i}' for i in range(n)]))
    X = np.concatenate(X_parts)
    y = np.concatenate(y_parts)
    run_ids = np.concatenate(id_parts)
    print(f'Total: X={X.shape}  Tumor={int(y.sum()):,}  Normal={int((y==0).sum()):,}')
    return X, y, run_ids

# ── Inline: seq_to_int ────────────────────────────────────────────────
_ENC = {'A': 1, 'C': 2, 'G': 3, 'T': 4, 'N': 5}
def seq_to_int(seq, max_len=80):
    nums = [_ENC.get(b, 5) for b in seq[:max_len]]
    if len(nums) < max_len:
        nums += [0] * (max_len - len(nums))
    return nums

# ── Inline: encode_fastq_file ─────────────────────────────────────────
def encode_fastq_file(fastq_path, label, run_id, max_reads=50_000, max_len=80):
    from Bio import SeqIO
    import gzip
    X, y, ids = [], [], []
    opener = gzip.open if fastq_path.endswith('.gz') else open
    with opener(fastq_path, 'rt') as f:
        try:
            for rec in SeqIO.parse(f, 'fastq'):
                X.append(seq_to_int(str(rec.seq), max_len))
                y.append(label)
                ids.append(run_id)
                if len(X) >= max_reads:
                    break
        except (EOFError, ValueError):
            # Truncated gzip stream (we only fetched a byte-range from ENA) —
            # the records parsed up to this point are still valid and complete.
            pass
    return X, y, ids

# ── Inline: class_balance_report ─────────────────────────────────────
def class_balance_report(batch_dir, cancer_label='Cancer'):
    X, y, _ = load_all_batches(batch_dir)
    n_tumor, n_normal, total = int(y.sum()), int((y==0).sum()), len(y)
    print(f'\nCLASS BALANCE — {cancer_label}')
    print(f'  Total: {total:,}  Tumor: {n_tumor:,}  Normal: {n_normal:,}')
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.bar(['Normal', 'Tumor'], [n_normal, n_tumor], color=['#2ecc71', '#e74c3c'])
    ax.set_title(f'Class Balance\n{cancer_label}', fontsize=12)
    ax.set_ylabel('Read Count')
    for i, v in enumerate([n_normal, n_tumor]):
        ax.text(i, v + total * 0.005, f'{v:,}', ha='center', fontsize=10)
    plt.tight_layout()
    save_path = os.path.join(batch_dir, 'class_balance.png')
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_path}')

# ── Inline: zero_shot_eval ────────────────────────────────────────────
def zero_shot_eval(model_path, batch_dir, cancer_label='Cancer', save_dir=None):
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
    print(f'Loading model from {model_path}...')
    model = load_model(model_path)
    X, y, run_ids = load_all_batches(batch_dir)
    print('Running inference...')
    probs = model.predict(X, batch_size=2048, verbose=1).flatten()
    fpr, tpr, _ = roc_curve(y, probs)
    read_auc = auc(fpr, tpr)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y, (probs >= 0.5).astype(int), average='binary', zero_division=0)
    df = pd.DataFrame({'run_id': run_ids, 'prob': probs, 'label': y})
    pat = df.groupby('run_id').agg(
        patient_prob=('prob', 'mean'),
        patient_label=('label', lambda x: int(x.mode()[0])),
    ).reset_index()
    pat_auc = None
    if len(pat['patient_label'].unique()) > 1:
        fpr_p, tpr_p, _ = roc_curve(pat['patient_label'], pat['patient_prob'])
        pat_auc = auc(fpr_p, tpr_p)
    print(f'\nZERO-SHOT — {cancer_label}')
    print(f'  Read AUC   : {read_auc:.4f}')
    if pat_auc:
        print(f'  Patient AUC: {pat_auc:.4f}')
    print(f'  Precision  : {prec:.4f}  Recall: {rec:.4f}  F1: {f1:.4f}')
    fig, axes = plt.subplots(1, 2 if pat_auc else 1,
                             figsize=(13 if pat_auc else 6, 5), dpi=130)
    if pat_auc is None:
        axes = [axes]
    axes[0].plot(fpr, tpr, color='#e74c3c', lw=2, label=f'AUC={read_auc:.4f}')
    axes[0].plot([0,1],[0,1],'k--',lw=1)
    axes[0].set_title(f'Read-Level ROC\n{cancer_label}', fontsize=12)
    axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
    axes[0].legend(loc='lower right')
    if pat_auc:
        axes[1].plot(fpr_p, tpr_p, color='#2980b9', lw=2, label=f'AUC={pat_auc:.4f}')
        axes[1].plot([0,1],[0,1],'k--',lw=1)
        axes[1].set_title(f'Patient-Level ROC\n{cancer_label}', fontsize=12)
        axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
        axes[1].legend(loc='lower right')
    plt.tight_layout()
    if save_dir:
        path = os.path.join(save_dir, 'zero_shot_roc.png')
        plt.savefig(path, dpi=200, bbox_inches='tight')
        print(f'Saved: {path}')
    plt.show()
    return {'read_auc': read_auc, 'patient_auc': pat_auc, 'precision': prec, 'recall': rec, 'f1': f1}

# ── Inline: SRA table parser ──────────────────────────────────────────
def assign_labels_from_sra_table(table_path):
    df = pd.read_csv(table_path, sep=None, engine='python')
    df.columns = [c.strip() for c in df.columns]
    run_col = next((c for c in df.columns if c.lower() in ('run','run_id','sra_id')), None)
    if run_col is None:
        raise ValueError("SraRunTable has no 'Run' column.")
    text_cols = [c for c in df.columns if c.lower() in
                 ('sample_type','tissue_type','source_name','disease','tumor_normal','sample_description')]
    rows = []
    for _, row in df.iterrows():
        combined = ' '.join(str(row.get(c,'')) for c in text_cols).lower()
        if any(t in combined for t in ('tumor','cancer','malignant')):
            rows.append({'Run': row[run_col], 'Label': 1})
        elif any(t in combined for t in ('normal','healthy','adjacent')):
            rows.append({'Run': row[run_col], 'Label': 0})
    result = pd.DataFrame(rows)
    print(f'Parsed {len(result)} runs (tumor={result["Label"].sum()}, normal={(result["Label"]==0).sum()})')
    return result

# ── Inline: ENA direct-download helpers ───────────────────────────────
# We bypass the SRA toolkit entirely (fasterq-dump kept failing in Colab with
# toolkit/network-resolver errors across multiple versions). ENA mirrors every
# SRA run as plain HTTPS .fastq.gz files, so we range-fetch just enough
# compressed bytes to cover MAX_READS reads and parse incrementally — no
# vdb-config, no cloud resolver, no multi-GB full-run downloads.
import requests

def ena_fastq_url(run_accession):
    r = requests.get(
        'https://www.ebi.ac.uk/ena/portal/api/filereport',
        params={'accession': run_accession, 'result': 'read_run',
                'fields': 'run_accession,fastq_ftp', 'format': 'tsv'},
        timeout=60,
    )
    r.raise_for_status()
    lines = r.text.strip().splitlines()
    if len(lines) < 2 or not lines[1].split('\t')[-1]:
        raise ValueError(f'No ENA fastq_ftp entry for {run_accession}')
    first_url = lines[1].split('\t')[-1].split(';')[0]
    return 'https://' + first_url

def ena_partial_download(run_accession, out_path, max_bytes=60_000_000):
    url = ena_fastq_url(run_accession)
    headers = {'Range': f'bytes=0-{max_bytes - 1}'}
    with requests.get(url, headers=headers, stream=True, timeout=300) as r:
        r.raise_for_status()
        with open(out_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
    return out_path

# ── Cohort metadata ───────────────────────────────────────────────────
COHORT_METADATA = {
    'brca': {
        'cancer_label': 'Breast Invasive Carcinoma (BRCA)',
        'geo_accession': 'GSE48215', 'sra_project': 'SRP028580',
        'n_tumor': 25, 'n_normal': 25,
        'runs': {
            'SRR949537': 1, 'SRR949538': 1, 'SRR949539': 1, 'SRR949540': 1,
            'SRR949541': 0, 'SRR949542': 0, 'SRR949543': 0, 'SRR949544': 0,
        },
    },
    'luad': {
        'cancer_label': 'Lung Adenocarcinoma (LUAD)',
        'geo_accession': 'GSE40419', 'sra_project': 'SRP013469',
        'n_tumor': 17, 'n_normal': 13,
        'runs': {
            'SRR521456': 1, 'SRR521457': 1, 'SRR521458': 1, 'SRR521459': 1,
            'SRR521460': 0, 'SRR521461': 0, 'SRR521462': 0, 'SRR521463': 0,
        },
    },
}

# ── Configuration ─────────────────────────────────────────────────────
CANCER_TYPE    = 'luad'    # 'brca' or 'luad'
MAX_READS      = 50_000
CNN_MODEL_PATH = f'{BASE}/ML Models/BreastCancer_CNN_Model.keras'

meta       = COHORT_METADATA[CANCER_TYPE]
OUTPUT_DIR = f'{BASE}/results/multi_cancer/{CANCER_TYPE}'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Setup OK.')
print(f'Cohort  : {meta["cancer_label"]}')
print(f'GEO     : {meta["geo_accession"]}   SRA: {meta["sra_project"]}')
print(f'Expected: {meta["n_tumor"]} tumor + {meta["n_normal"]} normal')
print(f'Output  : {OUTPUT_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Setup OK.
Cohort  : Lung Adenocarcinoma (LUAD)
GEO     : GSE40419   SRA: SRP013469
Expected: 17 tumor + 13 normal
Output  : /content/drive/MyDrive/DNA_Anomaly_Detection/results/multi_cancer/luad


## Configuration
Set `CANCER_TYPE` to `'brca'` or `'luad'`. Run this notebook twice to process both.

## Step 1 — Get SRA Run List
**Option A (recommended):** Download the SraRunTable from NCBI Run Selector and upload it.  
**Option B:** Use the small hardcoded subset in `multi_cancer_loader.py` (8 samples).

For Option A:
1. Go to: https://www.ncbi.nlm.nih.gov/Traces/study/?acc=SRP028580 (BRCA) or https://www.ncbi.nlm.nih.gov/Traces/study/?acc=SRP013469 (LUAD)
2. Click **Metadata** → download `SraRunTable.txt`
3. Upload to this Colab session

In [ ]:
SRA_TABLE_PATH = None   # set to '/content/SraRunTable.txt' if you uploaded it

if SRA_TABLE_PATH and os.path.exists(SRA_TABLE_PATH):
    run_df = assign_labels_from_sra_table(SRA_TABLE_PATH)
    # Filter to WXS only
    if 'LibraryStrategy' in run_df.columns:
        run_df = run_df[run_df['LibraryStrategy'] == 'WXS']
    print(f'Using SraRunTable: {len(run_df)} runs')
else:
    # Fallback: hardcoded subset
    run_df = pd.DataFrame([
        {'Run': acc, 'Label': lbl}
        for acc, lbl in meta['runs'].items()
    ])
    print(f'Using hardcoded subset: {len(run_df)} runs')
    print('TIP: Upload SraRunTable.txt for the full cohort.')

print(f'\nRuns to download: {len(run_df)}')
print(f'  Tumor  : {(run_df["Label"]==1).sum()}')
print(f'  Normal : {(run_df["Label"]==0).sum()}')
display(run_df.head(10))

Using hardcoded subset: 8 runs
TIP: Upload SraRunTable.txt for the full cohort.

Runs to download: 8
  Tumor  : 4
  Normal : 4


,Run,Label
0,SRR521456,1
1,SRR521457,1
2,SRR521458,1
3,SRR521459,1
4,SRR521460,0
5,SRR521461,0
6,SRR521462,0
7,SRR521463,0


## Step 2 — Download FASTQ Files from ENA
Each run is range-fetched directly from the European Nucleotide Archive (ENA)
over plain HTTPS — only the first ~60 MB of compressed data per sample
(comfortably more than `MAX_READS` reads' worth), no SRA-toolkit required.
With 8 samples this takes roughly 2-5 minutes on Colab.

In [ ]:
# ── Download FASTQ data directly from ENA (no SRA toolkit needed) ────
# fasterq-dump repeatedly failed in Colab (unconfigured toolkit, missing -X
# support, "Failed to call external services" cloud-resolver errors across
# three different toolkit versions/installs). ENA mirrors every SRA run as
# plain gzipped FASTQ over HTTPS, so we just range-fetch the first ~60MB of
# each run — comfortably more than enough compressed data to cover
# MAX_READS reads — and parse the (possibly truncated) gzip stream
# incrementally in the preprocessing step.
FASTQ_DIR = f'/content/fastq_{CANCER_TYPE}'
os.makedirs(FASTQ_DIR, exist_ok=True)

print(f'Downloading {len(run_df)} FASTQ files (ENA, partial) to {FASTQ_DIR}...')
for i, row in run_df.iterrows():
    acc = row['Run']
    outfile = os.path.join(FASTQ_DIR, f'{acc}_1.fastq.gz')
    if os.path.exists(outfile) and os.path.getsize(outfile) > 0:
        print(f'  {acc} already downloaded, skipping.')
        continue
    print(f'  [{i+1}/{len(run_df)}] Downloading {acc} from ENA...')
    try:
        ena_partial_download(acc, outfile, max_bytes=60_000_000)
        size_mb = os.path.getsize(outfile) / 1e6
        print(f'    OK — {size_mb:.1f} MB fetched')
    except Exception as e:
        print(f'    WARNING: failed to download {acc}: {e}')
        if os.path.exists(outfile):
            os.remove(outfile)

fastq_files = [f for f in os.listdir(FASTQ_DIR) if f.endswith('.fastq.gz')]
print(f'\nDownloaded {len(fastq_files)} FASTQ files.')

  [1/8] Downloading SRR521456 from ENA...
    OK — 60.0 MB fetched
  [2/8] Downloading SRR521457 from ENA...
    OK — 60.0 MB fetched
  [3/8] Downloading SRR521458 from ENA...
    OK — 60.0 MB fetched
  [4/8] Downloading SRR521459 from ENA...
    OK — 60.0 MB fetched
  [5/8] Downloading SRR521460 from ENA...
    OK — 60.0 MB fetched
  [6/8] Downloading SRR521461 from ENA...
    OK — 60.0 MB fetched
  [7/8] Downloading SRR521462 from ENA...
    OK — 60.0 MB fetched
  [8/8] Downloading SRR521463 from ENA...
    OK — 60.0 MB fetched

Downloaded 8 FASTQ files.


## Step 3 — Preprocess: Integer-Encode & Save Batches

In [ ]:
print(f'Processing {len(run_df)} samples into integer-encoded batches...')
FASTQ_DIR_LOCAL = f'/content/fastq_{CANCER_TYPE}'
batch_num = 1
X_buf, y_buf, id_buf = [], [], []

for i, row in run_df.iterrows():
    acc   = row['Run']
    label = int(row['Label'])
    fastq_path = os.path.join(FASTQ_DIR_LOCAL, f'{acc}_1.fastq.gz')
    if not os.path.exists(fastq_path):
        print(f'  WARNING: {fastq_path} not found, skipping.')
        continue
    print(f'  Encoding {acc}  (label={label})...')
    X_r, y_r, id_r = encode_fastq_file(fastq_path, label, acc, max_reads=MAX_READS)
    print(f'    -> {len(X_r):,} reads')
    X_buf.extend(X_r); y_buf.extend(y_r); id_buf.extend(id_r)

if X_buf:
    save_path = os.path.join(OUTPUT_DIR, f'batch_{batch_num:03d}.npz')
    np.savez_compressed(save_path,
                        X=np.array(X_buf, dtype=np.int8),
                        y=np.array(y_buf, dtype=np.int8),
                        run_ids=np.array(id_buf, dtype='U20'))
    print(f'Saved {len(X_buf):,} reads → {save_path}')
else:
    print('No reads encoded. Check that FASTQ files downloaded correctly.')

Processing 8 samples into integer-encoded batches...
  Encoding SRR521456  (label=1)...
    -> 50,000 reads
  Encoding SRR521457  (label=1)...
    -> 50,000 reads
  Encoding SRR521458  (label=1)...
    -> 50,000 reads
  Encoding SRR521459  (label=1)...
    -> 50,000 reads
  Encoding SRR521460  (label=0)...
    -> 50,000 reads
  Encoding SRR521461  (label=0)...
    -> 50,000 reads
  Encoding SRR521462  (label=0)...
    -> 50,000 reads
  Encoding SRR521463  (label=0)...
    -> 50,000 reads
Saved 400,000 reads → /content/drive/MyDrive/DNA_Anomaly_Detection/results/multi_cancer/luad/batch_001.npz


## Step 4 — Class Balance Report

In [ ]:
if glob.glob(os.path.join(OUTPUT_DIR, 'batch_*.npz')):
    class_balance_report(OUTPUT_DIR, cancer_label=meta['cancer_label'])
else:
    print(f'No batch_*.npz files in {OUTPUT_DIR} yet — '
          f'Step 2/3 (download/preprocess) must succeed first. Skipping.')

Loading batch_001.npz...
Total: X=(400000, 80)  Tumor=200,000  Normal=200,000

CLASS BALANCE — Lung Adenocarcinoma (LUAD)
  Total: 400,000  Tumor: 200,000  Normal: 200,000
Saved: /content/drive/MyDrive/DNA_Anomaly_Detection/results/multi_cancer/luad/class_balance.png


## Step 5 — Zero-Shot CNN Evaluation
Evaluates the Semester 1 CNN (trained on WXS breast cancer) on the new cohort
**without any retraining** to measure cross-cancer transfer performance.

In [ ]:
if glob.glob(os.path.join(OUTPUT_DIR, 'batch_*.npz')):
    eval_dir = f'{OUTPUT_DIR}/zero_shot_eval'
    zero_shot_eval(
        model_path=CNN_MODEL_PATH,
        batch_dir=OUTPUT_DIR,
        cancer_label=meta['cancer_label'],
        save_dir=eval_dir,
    )
    print(f'\nZero-shot evaluation plots saved to: {eval_dir}')
else:
    print(f'No batch_*.npz files in {OUTPUT_DIR} yet — '
          f'Step 2/3 (download/preprocess) must succeed first. Skipping.')

Loading model from /content/drive/MyDrive/DNA_Anomaly_Detection/ML Models/BreastCancer_CNN_Model.keras...
Loading batch_001.npz...
Total: X=(400000, 80)  Tumor=200,000  Normal=200,000
Running inference...
196/196 ━━━━━━━━━━━━━━━━━━━━ 37s 186ms/step

ZERO-SHOT — Lung Adenocarcinoma (LUAD)
  Read AUC   : 0.5053
  Patient AUC: 0.5000
  Precision  : 0.5775  Recall: 0.0015  F1: 0.0030
Saved: /content/drive/MyDrive/DNA_Anomaly_Detection/results/multi_cancer/luad/zero_shot_eval/zero_shot_roc.png

Zero-shot evaluation plots saved to: /content/drive/MyDrive/DNA_Anomaly_Detection/results/multi_cancer/luad/zero_shot_eval


## Summary

| Metric | WXS Breast (Sem. 1) | New Cohort (zero-shot) |
|--------|--------------------|-----------------------|
| Read-level AUC | 0.6157 | *see output above* |
| Patient-level AUC | 0.9156 | *see output above* |

**Interpretation:**
- If zero-shot AUC > 0.55: the CNN learned transferable somatic mutation features
- If zero-shot AUC ≈ 0.50: the signal is cancer-type specific → fine-tuning needed
- Patient-level crowd-voting will still amplify any consistent signal